In [23]:
import warnings
from IPython.display import Markdown, display
warnings.filterwarnings("ignore")

In [24]:
import pdfplumber as pdf

In [25]:
filename = "Muhammad-pages.pdf"
dictionary_for_pages=[]
with pdf.open(f"../data/building/{filename}") as pdfFile:
    for pageNo,content in enumerate(pdfFile.pages):
        dictionary_for_pages.append({"pageNo":pageNo,"text" : content.extract_text()})
        


In [26]:
import re
chunks = []
chunkNumber = 1

heading = "Default Heading"
current_text = []
startPage = None
endPage = None

for singleDictionary in dictionary_for_pages:

    pageNo = singleDictionary["pageNo"]
    pageText = singleDictionary["text"]

    lines = pageText.split("\n")

    for line in lines:
        line = line.strip()

        if not line:
            continue

        cleaned_line = re.sub(r'\[\d+\]', '', line)
        words = cleaned_line.split()

        # heading detected
        if len(words) < 8 and not cleaned_line.endswith("."):

            if current_text:
                chunks.append({
                    "startPage": startPage,
                    "endPage": endPage,
                    "chunkNumber": chunkNumber,
                    "heading": heading,
                    "chunk_text": " ".join(current_text).strip()
                })

                chunkNumber += 1

            heading = line
            current_text = []

            # new chunk starts from current page
            startPage = pageNo

        else:
            # first line of chunk
            if startPage is None:
                startPage = pageNo

            # update every time text is added
            endPage = pageNo

            current_text.append(line)


# push last remaining chunk
if current_text:
    chunks.append({
        "startPage": startPage,
        "endPage": endPage,
        "chunkNumber": chunkNumber,
        "heading": heading,
        "chunk_text": " ".join(current_text).strip()
    })

In [27]:
from sentence_transformers import SentenceTransformer as ST
model = ST("all-MiniLM-L6-v2")

Loading weights: 100%|██████████████████████████████████| 103/103 [00:00<00:00, 380.66it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
chunksText=[]
for chunk in chunks:
    chunksText.append(chunk["chunk_text"])
chunksText
embeddings = model.encode(chunksText)

In [29]:
embeddings

array([[-0.01479094,  0.15283845,  0.00446697, ..., -0.03008716,
        -0.0549854 ,  0.00385174],
       [ 0.00460467,  0.15764512, -0.04094992, ..., -0.01765833,
        -0.02792246, -0.03608555],
       [-0.01667172,  0.03446221, -0.02355837, ..., -0.06727172,
        -0.04321225, -0.00227335],
       [-0.00614691,  0.05360625, -0.03890175, ..., -0.05852663,
         0.01440349, -0.04789002],
       [-0.03182285,  0.05032117, -0.0300909 , ..., -0.05687704,
         0.03145624, -0.03772489]], shape=(5, 384), dtype=float32)

In [30]:
import faiss 

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [31]:
import os
from dotenv import load_dotenv
from groq import Groq

load_dotenv("../../.env")
os.getenv("GROQ_API_KEY")
# how to create client groq thing
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)
print(client)

In [33]:
# query changing
def generate_better_query(user_query):
    user_query+="\n write a short hypothetical answer to this question"
    # send the query to groq LLM
    message = {"role": "user","content":user_query}
    response  = client.chat.completions.create(model="llama-3.3-70b-versatile",messages=[message])
    return response.choices[0].message.content

In [34]:
def ask(query, idx):
    message = (query + "\n Answer the above question,only using the text below ,and \
    say I don't know if not found ,and also if you do find an answer then also tell, \
    which page number and chunk number u used,if startpage and end page are different ,then tell both,otherwise only one ")
    
    for i in idx[0]:
        message +=f"Page Number = {chunks[i]["startPage"]}\n"
        message +=f"EndingPage Number = {chunks[i]["endPage"]}\n"
        message +=f"Chunk Number = {chunks[i]["chunkNumber"]}\n"
        message += chunks[i]["chunk_text"] + "\n"
    messages = [{"role": "user", "content": message}]
    response  = client.chat.completions.create(model="llama-3.3-70b-versatile",messages=messages)
    return response

In [35]:
def evaluate_query(query, use_hyde=True):
    search_text = generate_better_query(query) if use_hyde else query
    query_embedding = model.encode([search_text])
    distances, idx = index.search(query_embedding, 3)
    response = ask(query, idx)  
    return {
        "query": query,
        "used_hyde": use_hyde,
        "search_text": search_text,
        "retrieved_pages": [(chunks[i]["startPage"], chunks[i]["endPage"], chunks[i]["chunkNumber"]) for i in idx[0]],
        "answer": response.choices[0].message.content
    }

In [36]:
listOfQuestions = [
    "Whom did Quraysh send towards rabbis and for what purpose?",
    "What led towards gossip in Mecca?",
    "What caused Mohammed distress?",
    "What one word did the Prophet say to Uqba's plea?",
    "Who accused the Holy Prophet of narrating wrong lines, and who recorded this?",
    "What was rejected by Islamic scholars?",
    "What reasons were posited by western scholars?",
    "What is the meaning of the name Mohammed?",
    "Who withdrew his support and for what reason?",
    "Who was Banu Nawaful?"
]

results = []
for q in listOfQuestions:
    results.append(evaluate_query(q, use_hyde=True))
    results.append(evaluate_query(q, use_hyde=False))

In [37]:
for r in results:
    print(f"Q: {r['query']}")
    print(f"HyDE used: {r['used_hyde']}")
    print(f"Search text: {r['search_text']}")
    print(f"Retrieved (startPage, endPage, chunkNumber): {r['retrieved_pages']}")
    print(f"Answer: {r['answer']}")
    print("-" * 80)

Q: Whom did Quraysh send towards rabbis and for what purpose?
HyDE used: True
Search text: Let's assume a hypothetical scenario. Quraysh sent a delegation comprising of two men, likely including a respected elder such as Urwa ibn Masud or Abdullah ibn Abu Rabia, towards the rabbis in Medina. 

Their primary purpose was to verify the claims made by the Prophet Muhammad regarding his prophethood. Quraysh was skeptical about Muhammad's assertion that he was a prophet, and they wanted to cross-check his information and knowledge with that of the learned Jewish scholars.

They inquired about the characteristics and signs of a true prophet as foretold by their scriptures, hoping to either validate or invalidate Muhammad's claims. However, as the rabbis confirmed many of the prophesied signs in their scriptures matched the descriptions provided by Muhammad, Quraysh's delegation found themselves faced with the daunting reality that their long-standing skepticism might be unfounded. 

Though th

In [38]:
import json

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)